# Exploratory Data Analysis for NBA Lineup Prediction and Hidden Patterns

This notebook organizes the data exploration steps from the raw CSV file and then implements several additional views to uncover hidden patterns. We aim to understand:

- **Lineup stability and variation** over time
- **Frequency of player appearances** in different positions (home_0 to home_4)
- **Outcome analysis** by lineup
- **Common matchups:** which home players appear most frequently against certain away lineups

These insights can guide further feature engineering to improve our predictive model.

In [33]:
# Import necessary libraries
import pandas as pd
import numpy as np
import glob
import os

import matplotlib.pyplot as plt
import seaborn as sns

# For warnings
import warnings
warnings.filterwarnings('ignore')

# Set display options for Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

pd.option_context('display.max_rows', None, 'display.max_columns', None)

## 1. Load and Inspect the Raw Data

Here we load a sample raw CSV file (2007 season) and inspect its contents.

In [34]:
# Define the folder containing the CSV files (update the path as needed)
data_folder = "../data"  # Replace with your actual data folder path

# Load all CSV files from 2007 to 2015
csv_files = sorted(glob.glob(os.path.join(data_folder, "matchups-20*.csv")))
dfs = []
for file in csv_files:
    print(f"Loading {file}")
    df = pd.read_csv(file)
    dfs.append(df)

df_all_years = pd.concat(dfs, ignore_index=True)

print("Dataset loaded with shape:", df_all_years.shape)
display(df_all_years.head())

Loading ../data\matchups-2007.csv
Loading ../data\matchups-2008.csv
Loading ../data\matchups-2009.csv
Loading ../data\matchups-2010.csv
Loading ../data\matchups-2011.csv
Loading ../data\matchups-2012.csv
Loading ../data\matchups-2013.csv
Loading ../data\matchups-2014.csv
Loading ../data\matchups-2015.csv
Dataset loaded with shape: (236912, 53)


,game,season,home_team,away_team,starting_min,end_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,fga_home,fta_home,fgm_home,fga_2_home,fgm_2_home,fga_3_home,fgm_3_home,ast_home,blk_home,pf_home,reb_home,dreb_home,oreb_home,to_home,pts_home,pct_home,pct_2_home,pct_3_home,fga_visitor,fta_visitor,fgm_visitor,fga_2_visitor,fgm_2_visitor,fga_3_visitor,fgm_3_visitor,ast_visitor,blk_visitor,pf_visitor,reb_visitor,dreb_visitor,oreb_visitor,to_visitor,pts_visitor,pct_visitor,pct_2_visitor,pct_3_visitor,outcome
0,200610310LAL,2007,LAL,PHO,0,5,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,10,4,4,7,3,3,1,3,0,1,1,1,0,1,9,0.400000,0.428571,0.333333,11,1,10,8,8,3,2,6,0,0,7,7,0,1,22,0.909091,1.00,0.666667,-1
1,200610310LAL,2007,LAL,PHO,6,7,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,3,0,2,2,2,1,0,1,0,0,2,2,0,2,4,0.666667,1.000000,0.000000,6,0,4,4,3,2,1,4,0,0,1,1,0,0,9,0.666667,0.75,0.500000,-1
2,200610310LAL,2007,LAL,PHO,8,9,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,4,0,2,4,2,0,0,2,1,1,2,1,1,2,4,0.500000,0.500000,0.000000,2,2,1,2,1,0,0,1,0,0,1,1,0,1,2,0.500000,0.50,0.000000,1
3,200610310LAL,2007,LAL,PHO,10,10,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,2,0,2,2,2,0,0,1,0,0,0,0,0,0,4,1.000000,1.000000,0.000000,2,0,1,0,0,2,1,1,0,0,1,0,1,1,3,0.500000,0.00,0.500000,1
4,200610310LAL,2007,LAL,PHO,11,11,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Vladimir Radmanovic,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,2,0,1,2,1,0,0,1,0,1,0,0,0,0,2,0.500000,0.500000,0.000000,1,0,1,1,1,0,0,1,0,1,1,1,0,1,2,1.000000,1.00,0.000000,-1


## 2. Filter to Allowed Features and Create a Date Column

We will filter the data to include only the allowed features and then extract a proper date from the `game` column.

In [35]:
# Define the allowed features
allowed_features = [
    'game', 'season', 'home_team', 'away_team', 'starting_min',
    'home_0', 'home_1', 'home_2', 'home_3', 'home_4',
    'away_0', 'away_1', 'away_2', 'away_3', 'away_4',
    'outcome'  
]

# Filter the dataframe
df_filtered = df_all_years[allowed_features].copy()

# Create a date column from the first 8 characters of the 'game' column
df_filtered['date'] = pd.to_datetime(df_filtered['game'].str[:8], format='%Y%m%d')

df_filtered

,game,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,date
0,200610310LAL,2007,LAL,PHO,0,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31
1,200610310LAL,2007,LAL,PHO,6,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31
2,200610310LAL,2007,LAL,PHO,8,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,1,2006-10-31
3,200610310LAL,2007,LAL,PHO,10,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,1,2006-10-31
4,200610310LAL,2007,LAL,PHO,11,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Vladimir Radmanovic,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,-1,2006-10-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236907,201504050NYK,2015,NYK,PHI,35,Jason Smith,Quincy Acy,Ricky Ledo,Shane Larkin,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05
236908,201504050NYK,2015,NYK,PHI,39,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05
236909,201504050NYK,2015,NYK,PHI,40,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Furkan Aldemir,Hollis Thompson,JaKarr Sampson,Jason Richardson,Nerlens Noel,-1,2015-04-05
236910,201504050NYK,2015,NYK,PHI,42,Andrea Bargnani,Jason Smith,Lance Thomas,Langston Galloway,Shane Larkin,Furkan Aldemir,Ish Smith,Jerami Grant,Nerlens Noel,Robert Covington,-1,2015-04-05


## 3. Create Lineup Tuples for Home and Away Teams

To explore lineup frequency and uniqueness, we create sorted tuples of players for both the home and away teams.

In [36]:
# Create lineup tuples (sorted alphabetically for consistency)
df_filtered['home_lineup'] = df_filtered[['home_0', 'home_1', 'home_2', 'home_3', 'home_4']].apply(lambda x: tuple(sorted(x)), axis=1)
df_filtered['away_lineup'] = df_filtered[['away_0', 'away_1', 'away_2', 'away_3', 'away_4']].apply(lambda x: tuple(sorted(x)), axis=1)

df_filtered[['game', 'home_lineup', 'away_lineup', 'date']]

,game,home_lineup,away_lineup,date
0,200610310LAL,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)",2006-10-31
1,200610310LAL,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",2006-10-31
2,200610310LAL,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",2006-10-31
3,200610310LAL,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",2006-10-31
4,200610310LAL,"(Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker, Vladimir Radmanovic)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",2006-10-31
...,...,...,...,...
236907,201504050NYK,"(Jason Smith, Quincy Acy, Ricky Ledo, Shane Larkin, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)",2015-04-05
236908,201504050NYK,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)",2015-04-05
236909,201504050NYK,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Furkan Aldemir, Hollis Thompson, JaKarr Sampson, Jason Richardson, Nerlens Noel)",2015-04-05
236910,201504050NYK,"(Andrea Bargnani, Jason Smith, Lance Thomas, Langston Galloway, Shane Larkin)","(Furkan Aldemir, Ish Smith, Jerami Grant, Nerlens Noel, Robert Covington)",2015-04-05


## 4. Unique Lineup Counts per Game

We group by game and count how many unique home and away lineups were used.

In [5]:
# Count unique lineups per game
lineup_counts = df_filtered.groupby('game').agg(
    unique_home_lineups=('home_lineup', 'nunique'),
    unique_away_lineups=('away_lineup', 'nunique')
).reset_index()

lineup_counts

,game,unique_home_lineups,unique_away_lineups
0,200610310LAL,11,9
1,200610310MIA,11,16
2,200611010BOS,19,17
3,200611010CHA,16,15
4,200611010CLE,12,10
...,...,...,...
10823,201504150MIN,15,14
10824,201504150NOP,14,16
10825,201504150NYK,12,14
10826,201504150PHI,9,2


In [37]:
unique_games = df_all_years['game'].nunique()
print("Total unique games:", unique_games)

Total unique games: 10828


In [38]:
df_filtered

,game,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,date,home_lineup,away_lineup
0,200610310LAL,2007,LAL,PHO,0,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Boris Diaw,Kurt Thomas,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)"
1,200610310LAL,2007,LAL,PHO,6,Andrew Bynum,Lamar Odom,Luke Walton,Sasha Vujacic,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,-1,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)"
2,200610310LAL,2007,LAL,PHO,8,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Amar'e Stoudemire,Leandro Barbosa,Raja Bell,Shawn Marion,Steve Nash,1,2006-10-31,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)"
3,200610310LAL,2007,LAL,PHO,10,Lamar Odom,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,1,2006-10-31,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)"
4,200610310LAL,2007,LAL,PHO,11,Luke Walton,Maurice Evans,Ronny Turiaf,Smush Parker,Vladimir Radmanovic,Boris Diaw,James Jones,Kurt Thomas,Leandro Barbosa,Marcus Banks,-1,2006-10-31,"(Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker, Vladimir Radmanovic)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236907,201504050NYK,2015,NYK,PHI,35,Jason Smith,Quincy Acy,Ricky Ledo,Shane Larkin,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05,"(Jason Smith, Quincy Acy, Ricky Ledo, Shane Larkin, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)"
236908,201504050NYK,2015,NYK,PHI,39,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Henry Sims,Hollis Thompson,JaKarr Sampson,Jason Richardson,Jerami Grant,-1,2015-04-05,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)"
236909,201504050NYK,2015,NYK,PHI,40,Jason Smith,Langston Galloway,Quincy Acy,Ricky Ledo,Tim Hardaway,Furkan Aldemir,Hollis Thompson,JaKarr Sampson,Jason Richardson,Nerlens Noel,-1,2015-04-05,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Furkan Aldemir, Hollis Thompson, JaKarr Sampson, Jason Richardson, Nerlens Noel)"
236910,201504050NYK,2015,NYK,PHI,42,Andrea Bargnani,Jason Smith,Lance Thomas,Langston Galloway,Shane Larkin,Furkan Aldemir,Ish Smith,Jerami Grant,Nerlens Noel,Robert Covington,-1,2015-04-05,"(Andrea Bargnani, Jason Smith, Lance Thomas, Langston Galloway, Shane Larkin)","(Furkan Aldemir, Ish Smith, Jerami Grant, Nerlens Noel, Robert Covington)"


## 5. Most Frequently Used Lineups per Game

We now identify the most frequently used home and away lineups for each game.

In [8]:
# Count occurrences of each lineup combination
lineup_usage = df_filtered.groupby(['game', 'home_team', 'away_team', 'date', 'home_lineup', 'away_lineup']).size().reset_index(name='count')

# For each game, select the lineup combination with the highest count
most_used_lineups = lineup_usage.sort_values(['game', 'count'], ascending=[True, False])

most_used_lineups.head()

,game,home_team,away_team,date,home_lineup,away_lineup,count
2,200610310LAL,LAL,PHO,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Maurice Evans, Smush Parker)","(Boris Diaw, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",2
3,200610310LAL,LAL,PHO,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Maurice Evans, Smush Parker)","(Kurt Thomas, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",2
8,200610310LAL,LAL,PHO,2006-10-31,"(Brian Cook, Jordan Farmar, Lamar Odom, Sasha Vujacic, Vladimir Radmanovic)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",2
0,200610310LAL,LAL,PHO,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Maurice Evans, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",1
1,200610310LAL,LAL,PHO,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Maurice Evans, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)",1


## 6. Additional Views and Analyses

In this section, we implement several new views to uncover hidden patterns:

1. **Frequency Analysis by Player Position:** How often does each player appear in a given home position (home_0, home_1, etc.)?
2. **Outcome Analysis by Home Lineup:** What are the win/loss outcomes (or average outcome) for different home lineups?
3. **Common Home Players Against Specific Away Lineups:** Which home players are most frequently used against a given away lineup?
4. **Lineup Variation Over Time:** How does the number of unique lineups change over time?

### 6.1 Frequency Analysis by Player Position

In [39]:
# Analyze frequency of players in each home position
for pos in ['home_0', 'home_1', 'home_2', 'home_3', 'home_4']:
    print(f"Frequency for {pos}:")
    print(df_filtered[pos].value_counts().head(10))
    print("---")

Frequency for home_0:
home_0
Andre Iguodala       5150
Al Jefferson         4492
Amar'e Stoudemire    3804
Boris Diaw           3502
Al Horford           3340
Al Harrington        3129
Beno Udrih           2951
Andrew Bogut         2652
Anderson Varejao     2650
Andray Blatche       2532
Name: count, dtype: int64
---
Frequency for home_1:
home_1
Dwyane Wade        2468
Carmelo Anthony    2446
David West         2433
Dirk Nowitzki      2258
Dwight Howard      2216
Andre Miller       2139
Deron Williams     2029
David Lee          1985
Kevin Durant       1983
Jamal Crawford     1982
Name: count, dtype: int64
---
Frequency for home_2:
home_2
LaMarcus Aldridge    2340
Josh Smith           2114
LeBron James         1864
J.R. Smith           1846
Kobe Bryant          1821
Jarrett Jack         1815
Deron Williams       1763
Paul Pierce          1761
Lamar Odom           1721
Mike Conley          1706
Name: count, dtype: int64
---
Frequency for home_3:
home_3
Tim Duncan       3281
Monta Ellis 

### 6.2 Outcome Analysis by Home Lineup

We group by the home lineup and calculate the average outcome and the number of games for each unique lineup. (Note: The `outcome` column typically indicates win/loss; you may need to adjust if using different metrics.)

In [89]:
# Outcome analysis by home lineup with home team and season information
lineup_outcomes = df_filtered.groupby(['season', 'home_team', 'home_lineup']).agg(
    games=('game', 'count'),
    avg_outcome=('outcome', 'mean')
).reset_index()

# Display the top 10 lineups by number of games played
top_lineups = lineup_outcomes.sort_values('games', ascending=False)




In [94]:
lineup_outcomes.head(35).sort_values('games', ascending=False)

,season,home_team,home_lineup,games,avg_outcome
16,2007,ATL,"(Anthony Johnson, Josh Childress, Josh Smith, Marvin Williams, Zaza Pachulia)",41,0.073171
12,2007,ATL,"(Anthony Johnson, Josh Childress, Josh Smith, Marvin Williams, Salim Stoudamire)",7,-0.428571
4,2007,ATL,"(Anthony Johnson, Joe Johnson, Josh Childress, Josh Smith, Zaza Pachulia)",6,-0.333333
22,2007,ATL,"(Anthony Johnson, Josh Childress, Josh Smith, Tyronn Lue, Zaza Pachulia)",5,-0.600000
14,2007,ATL,"(Anthony Johnson, Josh Childress, Josh Smith, Marvin Williams, Solomon Jones)",5,-1.000000
18,2007,ATL,"(Anthony Johnson, Josh Childress, Josh Smith, Salim Stoudamire, Zaza Pachulia)",4,0.500000
11,2007,ATL,"(Anthony Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Marvin Williams)",4,0.000000
9,2007,ATL,"(Anthony Johnson, Joe Johnson, Josh Childress, Marvin Williams, Zaza Pachulia)",4,-0.500000
29,2007,ATL,"(Anthony Johnson, Josh Childress, Marvin Williams, Solomon Jones, Tyronn Lue)",3,-1.000000
2,2007,ATL,"(Anthony Johnson, Joe Johnson, Josh Childress, Josh Smith, Marvin Williams)",3,-1.000000


In [95]:
lineup_outcomes.count()

season         60036
home_team      60036
home_lineup    60036
games          60036
avg_outcome    60036
dtype: int64

### 6.3 Most Common Home Players Against Specific Away Lineups

For each unique away lineup, we analyze which home players appear most frequently. Here we melt the home player columns into one long format and then group by away lineup and player.

In [97]:
# Melt the home player columns to analyze individual player frequency per team per season
home_players = df_filtered.melt(
    id_vars=['season', 'game', 'home_team', 'away_lineup', 'date'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='home_position',
    value_name='home_player'
)

# Count the frequency of each home player per away lineup, per team, and per season
common_players_vs_away = home_players.groupby(['season', 'home_team', 'away_lineup', 'home_player']).size().reset_index(name='count')

# Sort by frequency of occurrence
common_players_vs_away = common_players_vs_away.sort_values(['season', 'home_team', 'away_lineup', 'count'], ascending=[True, True, True, False])




In [98]:
common_players_vs_away

,season,home_team,away_lineup,home_player,count
1,2007,ATL,"(Aaron Williams, Chris Kaman, Cuttino Mobley, Daniel Ewing, Tim Thomas)",Josh Childress,5
2,2007,ATL,"(Aaron Williams, Chris Kaman, Cuttino Mobley, Daniel Ewing, Tim Thomas)",Lorenzen Wright,5
4,2007,ATL,"(Aaron Williams, Chris Kaman, Cuttino Mobley, Daniel Ewing, Tim Thomas)",Salim Stoudamire,4
7,2007,ATL,"(Aaron Williams, Chris Kaman, Cuttino Mobley, Daniel Ewing, Tim Thomas)",Zaza Pachulia,4
3,2007,ATL,"(Aaron Williams, Chris Kaman, Cuttino Mobley, Daniel Ewing, Tim Thomas)",Marvin Williams,3
...,...,...,...,...,...
745793,2015,WAS,"(Langston Galloway, Lou Amundson, Quincy Acy, Ricky Ledo, Shane Larkin)",Bradley Beal,1
745794,2015,WAS,"(Langston Galloway, Lou Amundson, Quincy Acy, Ricky Ledo, Shane Larkin)",John Wall,1
745795,2015,WAS,"(Langston Galloway, Lou Amundson, Quincy Acy, Ricky Ledo, Shane Larkin)",Kris Humphries,1
745796,2015,WAS,"(Langston Galloway, Lou Amundson, Quincy Acy, Ricky Ledo, Shane Larkin)",Marcin Gortat,1


In [43]:
# Melt the home player columns to analyze individual player frequency
home_players = df_filtered.melt(
    id_vars=['game', 'away_lineup', 'date'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='home_position',
    value_name='home_player'
)

# Count the frequency of each home player per away lineup
common_players_vs_away = home_players.groupby(['away_lineup', 'home_player']).size().reset_index(name='count')

# Sort the entire DataFrame by away_lineup and count descending
sorted_common_players = common_players_vs_away.sort_values(['away_lineup', 'count'], ascending=[True, False])

# Display the entire sorted DataFrame
sorted_common_players.head(35)


,away_lineup,home_player,count
0,"(A.J. Price, Alex Len, Archie Goodwin, Eric Bledsoe, Marcus Morris)",Corey Brewer,2
2,"(A.J. Price, Alex Len, Archie Goodwin, Eric Bledsoe, Marcus Morris)",Jason Terry,2
4,"(A.J. Price, Alex Len, Archie Goodwin, Eric Bledsoe, Marcus Morris)",Pablo Prigioni,2
5,"(A.J. Price, Alex Len, Archie Goodwin, Eric Bledsoe, Marcus Morris)",Trevor Ariza,2
1,"(A.J. Price, Alex Len, Archie Goodwin, Eric Bledsoe, Marcus Morris)",Donatas Motiejunas,1
3,"(A.J. Price, Alex Len, Archie Goodwin, Eric Bledsoe, Marcus Morris)",Josh Smith,1
6,"(A.J. Price, Alex Len, Archie Goodwin, P.J. Tucker, T.J. Warren)",Arron Afflalo,1
7,"(A.J. Price, Alex Len, Archie Goodwin, P.J. Tucker, T.J. Warren)",CJ McCollum,1
8,"(A.J. Price, Alex Len, Archie Goodwin, P.J. Tucker, T.J. Warren)",Chris Kaman,1
9,"(A.J. Price, Alex Len, Archie Goodwin, P.J. Tucker, T.J. Warren)",Dorell Wright,1


## 7. Conclusions and Next Steps

We have now organized the initial exploratory code and extended our analysis with additional views:

- **Frequency by Position:** Reveals which players appear most frequently in each home position.
- **Outcome Analysis:** Shows the performance (win/loss average) of different home lineups.
- **Common Matchups:** Identifies which home players are used most often against particular away lineups.
- **Temporal Trends:** Examines lineup variation over time.

These insights can help guide further feature engineering—such as adding a "lineup stability" score, network centrality measures for player synergy, or clustering of lineups—to improve the accuracy, precision, and recall of our missing-player prediction model.

Feel free to extend the analysis by exploring additional correlations (e.g., linking player positions if that information is available externally) or by applying association rule mining to detect frequent player combinations.

Let's continue discussing how we can further manipulate these data tables to gain even deeper insights.

In [ ]:
lineup_outcomes.sort_values('games', ascending=False).head(10)

,home_team,home_lineup,games,avg_outcome
3766,BOS,"(Kendrick Perkins, Kevin Garnett, Paul Pierce, Rajon Rondo, Ray Allen)",524,0.045802
21292,IND,"(David West, George Hill, Lance Stephenson, Paul George, Roy Hibbert)",399,-0.007519
40198,OKC,"(Kendrick Perkins, Kevin Durant, Russell Westbrook, Serge Ibaka, Thabo Sefolosha)",345,-0.113043
546,ATL,"(Al Horford, Joe Johnson, Josh Smith, Marvin Williams, Mike Bibby)",334,-0.017964
26920,MEM,"(Marc Gasol, Mike Conley, O.J. Mayo, Rudy Gay, Zach Randolph)",316,-0.101266
40117,OKC,"(Jeff Green, Kevin Durant, Nenad Krstic, Russell Westbrook, Thabo Sefolosha)",294,-0.095238
46731,POR,"(Damian Lillard, LaMarcus Aldridge, Nicolas Batum, Robin Lopez, Wesley Matthews)",272,-0.095588
22554,LAC,"(Blake Griffin, Chris Paul, DeAndre Jordan, J.J. Redick, Matt Barnes)",263,-0.011407
24320,LAL,"(Derek Fisher, Kobe Bryant, Lamar Odom, Metta World Peace, Pau Gasol)",241,-0.219917
23633,LAL,"(Andrew Bynum, Derek Fisher, Kobe Bryant, Metta World Peace, Pau Gasol)",223,-0.219731


In [56]:
# Extract the starting lineup for each game (row with the smallest starting_min per game)
starting_lineups = df_filtered.sort_values('starting_min').groupby(['season', 'game']).first().reset_index()

# Group by season, home_team, and home_lineup to compute number of games and the average outcome
starting_lineup_stats = starting_lineups.groupby(['season', 'home_team', 'home_lineup']).agg(
    games_count=('game', 'count'),
    avg_outcome=('outcome', 'mean')
).reset_index()

# For each season and home_team, select the lineup with the maximum games_count (most frequently used lineup)
idx = starting_lineup_stats.groupby(['season', 'home_team'])['games_count'].idxmax()
most_common_starting_lineups = starting_lineup_stats.loc[idx].reset_index(drop=True)

# Sort results to see the most used lineups first
most_common_starting_lineups = most_common_starting_lineups.sort_values(['season', 'games_count'], ascending=[True, False])

# Display the result
print(most_common_starting_lineups)


     season home_team  \
16     2007       MIN   
5      2007       DAL   
20     2007       ORL   
22     2007       PHO   
3      2007       CHI   
..      ...       ...   
261    2015       ORL   
257    2015       MIN   
259    2015       NYK   
244    2015       CHO   
262    2015       PHI   

                                                                                     home_lineup  \
16                        (Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)   
5                         (Devin Harris, Dirk Nowitzki, Erick Dampier, Jason Terry, Josh Howard)   
20                        (Dwight Howard, Grant Hill, Hedo Turkoglu, Jameer Nelson, Tony Battie)   
22                          (Amar'e Stoudemire, Boris Diaw, Raja Bell, Shawn Marion, Steve Nash)   
3                                 (Ben Gordon, Ben Wallace, Kirk Hinrich, Luol Deng, P.J. Brown)   
..                                                                                           ...   

In [78]:
most_common_starting_lineups.sort_values('season', ascending=True)

,season,home_team,home_lineup,games_count,avg_outcome
16,2007,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",26,-0.076923
5,2007,DAL,"(Devin Harris, Dirk Nowitzki, Erick Dampier, Jason Terry, Josh Howard)",23,0.304348
20,2007,ORL,"(Dwight Howard, Grant Hill, Hedo Turkoglu, Jameer Nelson, Tony Battie)",23,0.217391
22,2007,PHO,"(Amar'e Stoudemire, Boris Diaw, Raja Bell, Shawn Marion, Steve Nash)",22,0.363636
3,2007,CHI,"(Ben Gordon, Ben Wallace, Kirk Hinrich, Luol Deng, P.J. Brown)",20,0.400000
...,...,...,...,...,...
264,2015,POR,"(Damian Lillard, LaMarcus Aldridge, Nicolas Batum, Robin Lopez, Wesley Matthews)",16,-0.250000
266,2015,SAS,"(Danny Green, Kawhi Leonard, Tiago Splitter, Tim Duncan, Tony Parker)",15,0.866667
265,2015,SAC,"(Ben McLemore, Darren Collison, DeMarcus Cousins, Jason Thompson, Rudy Gay)",15,0.200000
241,2015,BOS,"(Avery Bradley, Brandon Bass, Evan Turner, Marcus Smart, Tyler Zeller)",14,0.428571


In [102]:
# Group by season, home_team, and home_lineup to calculate the number of games and average outcome
team_lineup_success = df_filtered.groupby(['season', 'home_team', 'home_lineup']).agg(
    games=('game', 'count'),
    avg_outcome=('outcome', 'mean')
).reset_index()

# Optionally, filter out lineups with very few appearances to avoid outliers
min_games = 5  # Adjust threshold as needed
team_lineup_success = team_lineup_success[team_lineup_success['games'] >= min_games]

# For each season and home_team, select the lineup with the highest average outcome
idx = team_lineup_success.groupby(['season', 'home_team'])['avg_outcome'].idxmax()
most_successful_lineups = team_lineup_success.loc[idx].reset_index(drop=True)

# Sort results for easier reading (by season and success rate)
most_successful_lineups = most_successful_lineups.sort_values(['season', 'avg_outcome'], ascending=[True, False])

# Display the final DataFrame
print(most_successful_lineups)


     season home_team  \
12     2007       LAL   
24     2007       SAC   
3      2007       CHI   
27     2007       TOR   
1      2007       BOS   
..      ...       ...   
253    2015       LAL   
252    2015       LAC   
268    2015       UTA   
259    2015       NYK   
265    2015       SAC   

                                                                    home_lineup  \
12    (Kobe Bryant, Lamar Odom, Ronny Turiaf, Sasha Vujacic, Shammond Williams)   
24    (Brad Miller, Corliss Williamson, John Salmons, Kevin Martin, Mike Bibby)   
3           (Adrian Griffin, Ben Wallace, Chris Duhon, Kirk Hinrich, Luol Deng)   
27   (Chris Bosh, Joey Graham, Jorge Garbajosa, Jose Calderon, Morris Peterson)   
1       (Allan Ray, Gerald Green, Kevinn Pinkney, Leon Powe, Sebastian Telfair)   
..                                                                          ...   
253            (Ed Davis, Jeremy Lin, Jordan Hill, Kobe Bryant, Wesley Johnson)   
252          (Chris Paul, DeAndre J

In [103]:
pd.set_option('display.max_colwidth', None)
most_successful_lineups

,season,home_team,home_lineup,games,avg_outcome
12,2007,LAL,"(Kobe Bryant, Lamar Odom, Ronny Turiaf, Sasha Vujacic, Shammond Williams)",5,1.000000
24,2007,SAC,"(Brad Miller, Corliss Williamson, John Salmons, Kevin Martin, Mike Bibby)",5,1.000000
3,2007,CHI,"(Adrian Griffin, Ben Wallace, Chris Duhon, Kirk Hinrich, Luol Deng)",7,0.714286
27,2007,TOR,"(Chris Bosh, Joey Graham, Jorge Garbajosa, Jose Calderon, Morris Peterson)",7,0.714286
1,2007,BOS,"(Allan Ray, Gerald Green, Kevinn Pinkney, Leon Powe, Sebastian Telfair)",6,0.666667
...,...,...,...,...,...
253,2015,LAL,"(Ed Davis, Jeremy Lin, Jordan Hill, Kobe Bryant, Wesley Johnson)",12,0.500000
252,2015,LAC,"(Chris Paul, DeAndre Jordan, Glen Davis, J.J. Redick, Matt Barnes)",7,0.428571
268,2015,UTA,"(Dante Exum, Elijah Millsap, Rudy Gobert, Trevor Booker, Trey Burke)",10,0.400000
259,2015,NYK,"(Alexey Shved, Cole Aldrich, Jason Smith, Shane Larkin, Travis Wear)",6,0.333333


In [104]:
# Extract home players, including season information, and rename the team column
home_players = df_filtered.melt(
    id_vars=['game', 'season', 'home_team'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='position',
    value_name='player'
).rename(columns={'home_team': 'team'})

# Extract away players, including season information, and rename the team column
away_players = df_filtered.melt(
    id_vars=['game', 'season', 'away_team'],
    value_vars=['away_0', 'away_1', 'away_2', 'away_3', 'away_4'],
    var_name='position',
    value_name='player'
).rename(columns={'away_team': 'team'})

# Combine home and away players into one DataFrame
all_players = pd.concat(
    [home_players[['game', 'season', 'team', 'player']], 
     away_players[['game', 'season', 'team', 'player']]],
    ignore_index=True
)

# Group by team and season to get unique players for each team in each season
team_players = all_players.groupby(['team', 'season'])['player'].unique().reset_index()

# Optionally, convert the array of players to a sorted, comma-separated string for easier reading
team_players['players'] = team_players['player'].apply(lambda x: ', '.join(sorted(x)))
team_players = team_players[['team', 'season', 'players']]

team_players


,team,season,players
0,ATL,2007,"Anthony Johnson, Cedric Bozeman, Dijon Thompson, Esteban Batista, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Marvin Williams, Matt Freije, Royal Ivey, Salim Stoudamire, Shelden Williams, Solomon Jones, Speedy Claxton, Stanislav Medvedenko, Tyronn Lue, Zaza Pachulia"
1,ATL,2008,"Acie Law, Al Horford, Anthony Johnson, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Mario West, Marvin Williams, Mike Bibby, Salim Stoudamire, Shelden Williams, Solomon Jones, Tyronn Lue, Zaza Pachulia"
2,ATL,2009,"Acie Law, Al Horford, Joe Johnson, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Ronald Murray, Solomon Jones, Speedy Claxton, Thomas Gardner, Zaza Pachulia"
3,ATL,2010,"Al Horford, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Joe Smith, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Zaza Pachulia"
4,ATL,2011,"Al Horford, Damien Wilkins, Etan Thomas, Hilton Armstrong, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Jordan Crawford, Josh Powell, Josh Smith, Kirk Hinrich, Marvin Williams, Maurice Evans, Mike Bibby, Pape Sy, Zaza Pachulia"
...,...,...,...
265,WAS,2011,"Al Thornton, Alonzo Gee, Andray Blatche, Cartier Martin, Gilbert Arenas, Hamady N'Diaye, Hilton Armstrong, JaVale McGee, John Wall, Jordan Crawford, Josh Howard, Kevin Seraphin, Kirk Hinrich, Larry Owens, Lester Hudson, Maurice Evans, Mike Bibby, Mustafa Shakur, Nick Young, Othyus Jeffers, Rashard Lewis, Trevor Booker, Yi Jianlian"
266,WAS,2012,"Andray Blatche, Brian Cook, Cartier Martin, Chris Singleton, Edwin Ubiles, JaVale McGee, James Singleton, Jan Vesely, John Wall, Jordan Crawford, Kevin Seraphin, Maurice Evans, Morris Almond, Nene Hilario, Nick Young, Rashard Lewis, Roger Mason, Ronny Turiaf, Shelvin Mack, Trevor Booker"
267,WAS,2013,"A.J. Price, Bradley Beal, Cartier Martin, Chris Singleton, Earl Barron, Emeka Okafor, Garrett Temple, Jan Vesely, Jannero Pargo, Jason Collins, John Wall, Jordan Crawford, Kevin Seraphin, Martell Webster, Nene Hilario, Shaun Livingston, Shelvin Mack, Trevor Ariza, Trevor Booker"
268,WAS,2014,"Al Harrington, Andre Miller, Bradley Beal, Chris Singleton, Drew Gooden, Eric Maynor, Garrett Temple, Glen Rice, Jan Vesely, John Wall, Kevin Seraphin, Marcin Gortat, Martell Webster, Nene Hilario, Otto Porter, Trevor Ariza, Trevor Booker"


In [115]:
# Melt the home player columns to analyze player participation
home_players = df_filtered.melt(
    id_vars=['game', 'season', 'home_team'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='home_position',
    value_name='player'
).rename(columns={'home_team': 'team'})

# Melt the away player columns
away_players = df_filtered.melt(
    id_vars=['game', 'season', 'away_team'],
    value_vars=['away_0', 'away_1', 'away_2', 'away_3', 'away_4'],
    var_name='away_position',
    value_name='player'
).rename(columns={'away_team': 'team'})

# Combine both home and away players into one DataFrame
all_players = pd.concat(
    [home_players[['game', 'season', 'team', 'player']], 
     away_players[['game', 'season', 'team', 'player']]],
    ignore_index=True
)

# Count how many games each player appeared in for each team in each season
player_game_counts = all_players.groupby(['season', 'team', 'player'])['game'].nunique().reset_index()

# Rename the column for clarity
player_game_counts.rename(columns={'game': 'games_played'}, inplace=True)

# Sort the results for better readability
player_game_counts = player_game_counts.sort_values(['season', 'team', 'games_played'], ascending=[True, True, False])


player_game_counts

,season,team,player,games_played
13,2007,ATL,Shelden Williams,81
7,2007,ATL,Josh Smith,72
18,2007,ATL,Zaza Pachulia,72
8,2007,ATL,Lorenzen Wright,67
9,2007,ATL,Marvin Williams,64
...,...,...,...,...
4725,2015,WAS,Ramon Sessions,28
4713,2015,WAS,DeJuan Blair,26
4728,2015,WAS,Will Bynum,6
4716,2015,WAS,Glen Rice,5


In [105]:
# First, aggregate unique players by team and season without converting to string
team_players_df = all_players.groupby(['team', 'season'])['player'].unique().reset_index()

# Convert the 'player' column (which is an array) to a list for each row
team_players_df['player'] = team_players_df['player'].apply(list)

# Now, build a nested dictionary: {team: {season: [players]}}
rosters_dict = {}
for _, row in team_players_df.iterrows():
    team = row['team']
    season = row['season']
    players_list = row['player']
    if team not in rosters_dict:
        rosters_dict[team] = {}
    rosters_dict[team][season] = sorted(players_list)  # sorted for consistency

# Print the dictionary
print(rosters_dict)


{'ATL': {2007: ['Anthony Johnson', 'Cedric Bozeman', 'Dijon Thompson', 'Esteban Batista', 'Jeremy Richardson', 'Joe Johnson', 'Josh Childress', 'Josh Smith', 'Lorenzen Wright', 'Marvin Williams', 'Matt Freije', 'Royal Ivey', 'Salim Stoudamire', 'Shelden Williams', 'Solomon Jones', 'Speedy Claxton', 'Stanislav Medvedenko', 'Tyronn Lue', 'Zaza Pachulia'], 2008: ['Acie Law', 'Al Horford', 'Anthony Johnson', 'Jeremy Richardson', 'Joe Johnson', 'Josh Childress', 'Josh Smith', 'Lorenzen Wright', 'Mario West', 'Marvin Williams', 'Mike Bibby', 'Salim Stoudamire', 'Shelden Williams', 'Solomon Jones', 'Tyronn Lue', 'Zaza Pachulia'], 2009: ['Acie Law', 'Al Horford', 'Joe Johnson', 'Josh Smith', 'Mario West', 'Marvin Williams', 'Maurice Evans', 'Mike Bibby', 'Othello Hunter', 'Randolph Morris', 'Ronald Murray', 'Solomon Jones', 'Speedy Claxton', 'Thomas Gardner', 'Zaza Pachulia'], 2010: ['Al Horford', 'Jamal Crawford', 'Jason Collins', 'Jeff Teague', 'Joe Johnson', 'Joe Smith', 'Josh Smith', 'Mari

In [116]:
rosters_dict['TOR'][2015] 

['Amir Johnson',
 'Bruno Caboclo',
 'Chuck Hayes',
 'DeMar DeRozan',
 'Greg Stiemsma',
 'Greivis Vasquez',
 'James Johnson',
 'Jonas Valanciunas',
 'Kyle Lowry',
 'Landry Fields',
 'Lou Williams',
 'Lucas Nogueira',
 'Patrick Patterson',
 'Terrence Ross',
 'Tyler Hansbrough']

In [112]:
# Sort the DataFrame by game and starting_min
df_sorted = df_filtered.sort_values(['game', 'starting_min']).copy()

# Compute the duration for each lineup segment per game.
# For each game, duration = next starting_min - current starting_min.
# For the last segment in each game, duration = 48 - current starting_min.
df_sorted['duration'] = df_sorted.groupby('game')['starting_min'].transform(lambda x: x.shift(-1) - x)
df_sorted['duration'] = df_sorted['duration'].fillna(48 - df_sorted['starting_min'])

# Melt the home lineup columns so each row corresponds to a player's appearance in that lineup segment.
# Include 'season' as part of the id_vars.
home_players = df_sorted.melt(
    id_vars=['game', 'season', 'home_team', 'duration'],
    value_vars=['home_0', 'home_1', 'home_2', 'home_3', 'home_4'],
    var_name='position',
    value_name='player'
)

# Group by team, season, and player, summing the durations (which represent the minutes played)
player_minutes = home_players.groupby(['home_team', 'season', 'player'])['duration'].sum().reset_index()

# Rename the 'duration' column for clarity
player_minutes.rename(columns={'duration': 'total_minutes'}, inplace=True)

# Select the top 10 players with the most minutes played per team per season
top10_by_team_season = (
    player_minutes.sort_values(['home_team', 'season', 'total_minutes'], ascending=[True, True, False])
    .groupby(['home_team', 'season'])
    .apply(lambda x: x.nlargest(10, 'total_minutes'))
    .reset_index(drop=True)
)




In [113]:
top10_by_team_season.head(12)

,home_team,season,player,total_minutes
0,ATL,2007,Josh Smith,1322.0
1,ATL,2007,Joe Johnson,1120.0
2,ATL,2007,Marvin Williams,1107.0
3,ATL,2007,Josh Childress,1077.0
4,ATL,2007,Zaza Pachulia,1034.0
5,ATL,2007,Tyronn Lue,764.0
6,ATL,2007,Shelden Williams,720.0
7,ATL,2007,Lorenzen Wright,531.0
8,ATL,2007,Speedy Claxton,510.0
9,ATL,2007,Salim Stoudamire,488.0


In [120]:
# Step 1: Extract the starting lineup for each game (i.e., the row with the smallest starting_min per game)
starting_lineups = df_filtered.sort_values('starting_min').groupby('game').first().reset_index()

# Step 2: Merge with the most common starting lineups (most_common_starting_lineups)
# most_common_starting_lineups has columns: season, home_team, home_lineup, games_count, avg_outcome
common_starting_games = starting_lineups.merge(
    most_common_starting_lineups[['season', 'home_team', 'home_lineup']],
    on=['season', 'home_team', 'home_lineup'],
    how='inner'
)

# Step 3: Now, group by season, home_team, and away_team to compute the number of games and average outcome
team_vs_team_common_lineup = common_starting_games.groupby(['season', 'home_team', 'away_team']).agg(
    games_played=('game', 'count'),
    avg_outcome=('outcome', 'mean')
).reset_index()

# Step 4: Sort the results (by season, team, then average outcome descending)
team_vs_team_common_lineup = team_vs_team_common_lineup.sort_values(
    ['season', 'home_team', 'avg_outcome'], ascending=[True, True, False]
)




In [123]:
team_vs_team_common_lineup.head(30)

,season,home_team,away_team,games_played,avg_outcome
0,2007,ATL,CHI,1,1.0
2,2007,ATL,MEM,1,1.0
4,2007,ATL,MIN,1,1.0
5,2007,ATL,PHI,1,1.0
6,2007,ATL,POR,1,1.0
7,2007,ATL,SAC,1,1.0
1,2007,ATL,DAL,1,-1.0
3,2007,ATL,MIA,1,-1.0
8,2007,BOS,DET,1,1.0
9,2007,BOS,IND,1,1.0


In [61]:
from sklearn.model_selection import KFold
import pandas as pd

# Define individual player columns
player_columns = ['home_0', 'home_1', 'home_2', 'home_3', 'home_4',
                  'away_0', 'away_1', 'away_2', 'away_3', 'away_4']

# Create a copy of the dataset for encoding
df_encoded = df_filtered.copy()

# Initialize dictionary to store mean outcome values per player per team per season
player_mean_outcome = {}

# Apply K-Fold Cross-Validation for Mean Encoding (Avoids Data Leakage)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_index, val_index in kf.split(df_encoded):
    train_data, val_data = df_encoded.iloc[train_index], df_encoded.iloc[val_index]

    for col in player_columns:
        # Compute mean outcome per player within each team and season
        player_mean = train_data.groupby(['season', 'home_team', col])['outcome'].mean()

        # Store computed values
        player_mean_outcome.update(player_mean.to_dict())

        # Apply encoding to validation set
        val_data[col] = val_data.set_index(['season', 'home_team', col]).index.map(player_mean_outcome)

    # Update the main dataset with mean encoded values
    df_encoded.iloc[val_index] = val_data

# Fill NaN values for unseen players with:
# - Team-season win rate
# - Global average outcome across all seasons
df_encoded['team_season_win_rate'] = df_encoded.groupby(['season', 'home_team'])['outcome'].transform('mean')
overall_mean_outcome = df_encoded['outcome'].mean()

for col in player_columns:
    df_encoded[col] = df_encoded[col].fillna(df_encoded['team_season_win_rate']).fillna(overall_mean_outcome)

# Display the final dataset after encoding
print("Final Dataset After Mean Encoding:")
display(df_encoded)
print(df_encoded.info())


Final Dataset After Mean Encoding:


,game,season,home_team,away_team,starting_min,home_0,home_1,home_2,home_3,home_4,away_0,away_1,away_2,away_3,away_4,outcome,date,home_lineup,away_lineup,team_season_win_rate
0,200610310LAL,2007,LAL,PHO,0,-0.183486,0.052632,-0.065217,-0.142857,-0.120000,0.555556,-0.333333,0.200000,0.100000,-0.076923,-1,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Boris Diaw, Kurt Thomas, Raja Bell, Shawn Marion, Steve Nash)",-0.132723
1,200610310LAL,2007,LAL,PHO,6,-0.208333,0.192308,0.044776,-0.056604,-0.087500,0.200000,0.000000,0.333333,0.230769,0.200000,-1,2006-10-31,"(Andrew Bynum, Lamar Odom, Luke Walton, Sasha Vujacic, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",-0.132723
2,200610310LAL,2007,LAL,PHO,8,-0.083333,-0.142857,0.037037,0.314286,-0.135802,0.333333,0.000000,0.333333,0.500000,0.600000,1,2006-10-31,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Amar'e Stoudemire, Leandro Barbosa, Raja Bell, Shawn Marion, Steve Nash)",-0.132723
3,200610310LAL,2007,LAL,PHO,10,0.333333,-0.076923,0.056604,0.101449,-0.087500,0.500000,0.600000,0.500000,0.500000,1.000000,1,2006-10-31,"(Lamar Odom, Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",-0.132723
4,200610310LAL,2007,LAL,PHO,11,-0.132723,-0.384615,-0.241379,-0.296296,-0.180124,0.272727,0.166667,0.111111,0.200000,1.000000,-1,2006-10-31,"(Luke Walton, Maurice Evans, Ronny Turiaf, Smush Parker, Vladimir Radmanovic)","(Boris Diaw, James Jones, Kurt Thomas, Leandro Barbosa, Marcus Banks)",-0.132723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236907,201504050NYK,2015,NYK,PHI,35,-0.250000,-0.250000,1.000000,-0.260870,-0.312500,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1,2015-04-05,"(Jason Smith, Quincy Acy, Ricky Ledo, Shane Larkin, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)",-0.209063
236908,201504050NYK,2015,NYK,PHI,39,-0.314286,-0.200000,-0.222222,-0.250000,-0.328571,0.000000,-0.714286,-1.000000,-0.500000,-1.000000,-1,2015-04-05,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Henry Sims, Hollis Thompson, JaKarr Sampson, Jason Richardson, Jerami Grant)",-0.209063
236909,201504050NYK,2015,NYK,PHI,40,-0.314286,-0.200000,-0.222222,-0.250000,-0.328571,-0.500000,-0.714286,-1.000000,-0.500000,-0.500000,-1,2015-04-05,"(Jason Smith, Langston Galloway, Quincy Acy, Ricky Ledo, Tim Hardaway)","(Furkan Aldemir, Hollis Thompson, JaKarr Sampson, Jason Richardson, Nerlens Noel)",-0.209063
236910,201504050NYK,2015,NYK,PHI,42,-0.307692,-0.202532,-0.262136,-0.182796,-0.280000,-0.400000,0.142857,-0.428571,0.181818,-0.333333,-1,2015-04-05,"(Andrea Bargnani, Jason Smith, Lance Thomas, Langston Galloway, Shane Larkin)","(Furkan Aldemir, Ish Smith, Jerami Grant, Nerlens Noel, Robert Covington)",-0.209063


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236912 entries, 0 to 236911
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   game                  236912 non-null  object        
 1   season                236912 non-null  int64         
 2   home_team             236912 non-null  object        
 3   away_team             236912 non-null  object        
 4   starting_min          236912 non-null  int64         
 5   home_0                236912 non-null  float64       
 6   home_1                236912 non-null  float64       
 7   home_2                236912 non-null  float64       
 8   home_3                236912 non-null  float64       
 9   home_4                236912 non-null  float64       
 10  away_0                236912 non-null  float64       
 11  away_1                236912 non-null  float64       
 12  away_2                236912 non-null  float64       
 13 

In [117]:
top10_by_team_season

,home_team,season,player,total_minutes
0,ATL,2007,Josh Smith,1322.0
1,ATL,2007,Joe Johnson,1120.0
2,ATL,2007,Marvin Williams,1107.0
3,ATL,2007,Josh Childress,1077.0
4,ATL,2007,Zaza Pachulia,1034.0
...,...,...,...,...
2695,WAS,2015,Rasual Butler,712.0
2696,WAS,2015,Kris Humphries,708.0
2697,WAS,2015,Otto Porter,703.0
2698,WAS,2015,Kevin Seraphin,571.0


In [70]:
team_players

,team,season,players
0,ATL,2007,"Anthony Johnson, Cedric Bozeman, Dijon Thompson, Esteban Batista, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Marvin Williams, Matt Freije, Royal Ivey, Salim Stoudamire, Shelden Williams, Solomon Jones, Speedy Claxton, Stanislav Medvedenko, Tyronn Lue, Zaza Pachulia"
1,ATL,2008,"Acie Law, Al Horford, Anthony Johnson, Jeremy Richardson, Joe Johnson, Josh Childress, Josh Smith, Lorenzen Wright, Mario West, Marvin Williams, Mike Bibby, Salim Stoudamire, Shelden Williams, Solomon Jones, Tyronn Lue, Zaza Pachulia"
2,ATL,2009,"Acie Law, Al Horford, Joe Johnson, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Ronald Murray, Solomon Jones, Speedy Claxton, Thomas Gardner, Zaza Pachulia"
3,ATL,2010,"Al Horford, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Joe Smith, Josh Smith, Mario West, Marvin Williams, Maurice Evans, Mike Bibby, Othello Hunter, Randolph Morris, Zaza Pachulia"
4,ATL,2011,"Al Horford, Damien Wilkins, Etan Thomas, Hilton Armstrong, Jamal Crawford, Jason Collins, Jeff Teague, Joe Johnson, Jordan Crawford, Josh Powell, Josh Smith, Kirk Hinrich, Marvin Williams, Maurice Evans, Mike Bibby, Pape Sy, Zaza Pachulia"
...,...,...,...
265,WAS,2011,"Al Thornton, Alonzo Gee, Andray Blatche, Cartier Martin, Gilbert Arenas, Hamady N'Diaye, Hilton Armstrong, JaVale McGee, John Wall, Jordan Crawford, Josh Howard, Kevin Seraphin, Kirk Hinrich, Larry Owens, Lester Hudson, Maurice Evans, Mike Bibby, Mustafa Shakur, Nick Young, Othyus Jeffers, Rashard Lewis, Trevor Booker, Yi Jianlian"
266,WAS,2012,"Andray Blatche, Brian Cook, Cartier Martin, Chris Singleton, Edwin Ubiles, JaVale McGee, James Singleton, Jan Vesely, John Wall, Jordan Crawford, Kevin Seraphin, Maurice Evans, Morris Almond, Nene Hilario, Nick Young, Rashard Lewis, Roger Mason, Ronny Turiaf, Shelvin Mack, Trevor Booker"
267,WAS,2013,"A.J. Price, Bradley Beal, Cartier Martin, Chris Singleton, Earl Barron, Emeka Okafor, Garrett Temple, Jan Vesely, Jannero Pargo, Jason Collins, John Wall, Jordan Crawford, Kevin Seraphin, Martell Webster, Nene Hilario, Shaun Livingston, Shelvin Mack, Trevor Ariza, Trevor Booker"
268,WAS,2014,"Al Harrington, Andre Miller, Bradley Beal, Chris Singleton, Drew Gooden, Eric Maynor, Garrett Temple, Glen Rice, Jan Vesely, John Wall, Kevin Seraphin, Marcin Gortat, Martell Webster, Nene Hilario, Otto Porter, Trevor Ariza, Trevor Booker"


In [79]:
most_common_starting_lineups

,season,home_team,home_lineup,games_count,avg_outcome
16,2007,MIN,"(Kevin Garnett, Mark Blount, Mike James, Ricky Davis, Trenton Hassell)",26,-0.076923
5,2007,DAL,"(Devin Harris, Dirk Nowitzki, Erick Dampier, Jason Terry, Josh Howard)",23,0.304348
20,2007,ORL,"(Dwight Howard, Grant Hill, Hedo Turkoglu, Jameer Nelson, Tony Battie)",23,0.217391
22,2007,PHO,"(Amar'e Stoudemire, Boris Diaw, Raja Bell, Shawn Marion, Steve Nash)",22,0.363636
3,2007,CHI,"(Ben Gordon, Ben Wallace, Kirk Hinrich, Luol Deng, P.J. Brown)",20,0.400000
...,...,...,...,...,...
261,2015,ORL,"(Dewayne Dedmon, Elfrid Payton, Nikola Vucevic, Tobias Harris, Victor Oladipo)",7,-0.142857
257,2015,MIN,"(Andrew Wiggins, Corey Brewer, Gorgui Dieng, Thaddeus Young, Zach LaVine)",6,-0.666667
259,2015,NYK,"(Andrea Bargnani, Lance Thomas, Langston Galloway, Lou Amundson, Shane Larkin)",6,-0.666667
244,2015,CHO,"(Al Jefferson, Cody Zeller, Gerald Henderson, Kemba Walker, Lance Stephenson)",5,-0.600000
